In [ ]:
## Text tp metadata filter

import os
from langchain_groq import ChatGroq
from langchain_classic.chains.query_constructor.base import (
    AttributeInfo,
    load_query_constructor_runnable,
)


In [4]:
# Define the metadata fields available in your vector store
metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="The genre of the movie. One of ['science fiction', 'comedy', 'drama', 'thriller', 'romance', 'action']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the movie was released",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="The name of the movie director",
        type="string",
    ),
    AttributeInfo(
        name="rating", 
        description="A 1-10 rating for the movie", 
        type="float"
    ),
]


In [2]:
# Describe what the documents are actually about
document_content_description = "Brief summary of a movie"


In [11]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=os.environ.get("GROQ_API_KEY"),
)

In [13]:
# Create the query constructor runnable.
# In LangChain 1.x, legacy query-constructor functionality is provided by langchain-classic.
chain = load_query_constructor_runnable(
    llm=llm,
    document_contents=document_content_description,
    attribute_info=metadata_field_info,
    fix_invalid=True,
)


In [18]:
from pprint import pformat


def readable_filter(expression):
    if hasattr(expression, "operator") and hasattr(expression, "arguments"):
        return {
            "operator": expression.operator.value,
            "arguments": [readable_filter(argument) for argument in expression.arguments],
        }
    if hasattr(expression, "comparator"):
        return {
            "comparator": expression.comparator.value,
            "attribute": expression.attribute,
            "value": expression.value,
        }
    return expression

# Demonstration
def translate_query(query_text, query_number):
    result = chain.invoke({"query": query_text})
    content_query = result.query.strip() or "(none)"

    print(f"\n{'=' * 60}")
    print(f"Query {query_number}")
    print(f"{'=' * 60}")
    print(f"User's request:\n  {query_text}")
    print("\nContent search query:")
    print(f"  {content_query}")
    print("\nMetadata filter:")
    print(pformat(readable_filter(result.filter), indent=2, sort_dicts=False))

# Try it out
translate_query("What are some highly rated action movies from 1994?", 1)
translate_query("Find me comedies directed by Greta Gerwig", 2)
translate_query("Show me movies that aren't thrillers but have a rating above 8", 3)


Query 1
User's request:
  What are some highly rated action movies from 1994?

Content search query:
  (none)

Metadata filter:
{ 'operator': 'and',
  'arguments': [ {'comparator': 'eq', 'attribute': 'genre', 'value': 'action'},
                 {'comparator': 'eq', 'attribute': 'year', 'value': 1994},
                 {'comparator': 'gte', 'attribute': 'rating', 'value': 8}]}

Query 2
User's request:
  Find me comedies directed by Greta Gerwig

Content search query:
  (none)

Metadata filter:
{ 'operator': 'and',
  'arguments': [ {'comparator': 'eq', 'attribute': 'genre', 'value': 'comedy'},
                 { 'comparator': 'eq',
                   'attribute': 'director',
                   'value': 'Greta Gerwig'}]}

Query 3
User's request:
  Show me movies that aren't thrillers but have a rating above 8

Content search query:
  (none)

Metadata filter:
{ 'operator': 'and',
  'arguments': [ { 'comparator': 'ne',
                   'attribute': 'genre',
                   'value': '

In [21]:
translate_query("Moview from 2000 that are action or thriller", 1)


Query 1
User's request:
  Moview from 2000 that are action or thriller

Content search query:
  (none)

Metadata filter:
{ 'operator': 'and',
  'arguments': [ {'comparator': 'eq', 'attribute': 'year', 'value': 2000},
                 { 'operator': 'or',
                   'arguments': [ { 'comparator': 'eq',
                                    'attribute': 'genre',
                                    'value': 'action'},
                                  { 'comparator': 'eq',
                                    'attribute': 'genre',
                                    'value': 'thriller'}]}]}


## Text to SQL

Translating natural language to SQL queries using LangChain's SQLDatabaseChain.

In [24]:
from langchain_community.utilities import SQLDatabase
from langchain_experimental.sql import SQLDatabaseChain
from sqlalchemy import create_engine, Column, Integer, String, Float
from sqlalchemy.orm import declarative_base, sessionmaker

# 1. Setup an in-memory SQLite database for the example
engine = create_engine('sqlite:///:memory:')
Base = declarative_base()

class Movie(Base):
    __tablename__ = 'movies'
    id = Column(Integer, primary_key=True)
    title = Column(String)
    genre = Column(String)
    year = Column(Integer)
    director = Column(String)
    rating = Column(Float)

Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)
session = Session()

# Add some sample data
movies = [
    Movie(title='The Shawshank Redemption', genre='drama', year=1994, director='Frank Darabont', rating=9.3),
    Movie(title='Pulp Fiction', genre='thriller', year=1994, director='Quentin Tarantino', rating=8.9),
    Movie(title='The Dark Knight', genre='action', year=2008, director='Christopher Nolan', rating=9.0),
    Movie(title='Greta', genre='thriller', year=2018, director='Neil Jordan', rating=6.0),
    Movie(title='Barbie', genre='comedy', year=2023, director='Greta Gerwig', rating=7.0),
]
session.add_all(movies)
session.commit()

db = SQLDatabase(engine)
db_chain = SQLDatabaseChain.from_llm(llm, db, verbose=True)

def text_to_sql_demo(query):
    print(f'\nUser Request: {query}')
    try:
        result = db_chain.run(query)
        print(f'Result: {result}')
    except Exception as e:
        print(f'Error: {e}')

text_to_sql_demo('How many movies are there?')
text_to_sql_demo('What is the highest rated movie from 1994?')
text_to_sql_demo('List all comedy movies')

C:\Users\Yusuf Solomon\AppData\Local\Temp\ipykernel_3276\1104988068.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.sql import SQLDatabaseChain



User Request: How many movies are there?


> Entering new SQLDatabaseChain chain...
How many movies are there?
SQLQuery:

C:\Users\Yusuf Solomon\AppData\Local\Temp\ipykernel_3276\1104988068.py:40: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = db_chain.run(query)


Question: How many movies are there?
SQLQuery: SELECT COUNT(*) AS "movie_count" FROM "movies";
SQLResult: [(5,)]
Answer:
> Finished chain.
Result: 

User Request: What is the highest rated movie from 1994?


> Entering new SQLDatabaseChain chain...
What is the highest rated movie from 1994?
SQLQuery:
SQLResult: 
Answer:Question: What is the highest rated movie from 1994?
SQLQuery: SELECT "title", "rating" FROM "movies" WHERE "year" = 1994 ORDER BY "rating" DESC LIMIT 1;
> Finished chain.
Result: Question: What is the highest rated movie from 1994?
SQLQuery: SELECT "title", "rating" FROM "movies" WHERE "year" = 1994 ORDER BY "rating" DESC LIMIT 1;

User Request: List all comedy movies


> Entering new SQLDatabaseChain chain...
List all comedy movies
SQLQuery:
SQLResult: 
Answer:
> Finished chain.
Result: 


## Text to SQL + Semantic Search

This setup combines structured SQL-like filtering with semantic content search, similar to a hybrid search approach.

In [25]:
# Conceptual example of combining SQL (structured) and Vector (semantic)

def hybrid_query(query):
    print(f'\nHybrid Request: {query}')
    # 1. Structured part: Extract filters (like we did in the first section)
    # 2. Semantic part: Content search query
    
    result = chain.invoke({'query': query})
    
    content_query = result.query.strip() or "(none)"
    print(f'Content Query (Semantic): {content_query}')
    print(f'Metadata Filter (Structured):')
    from pprint import pprint
    pprint(readable_filter(result.filter))

hybrid_query('Find a movie about a bank heist that was released after 2010 with a rating above 8')


Hybrid Request: Find a movie about a bank heist that was released after 2010 with a rating above 8
Content Query (Semantic): bank heist
Metadata Filter (Structured):
{'arguments': [{'attribute': 'year', 'comparator': 'gt', 'value': 2010},
               {'attribute': 'rating', 'comparator': 'gt', 'value': 8}],
 'operator': 'and'}


In [27]:
hybrid_query('Find a movie about an apocalypse that was released after 2010 where the main character is a woman')


Hybrid Request: Find a movie about an apocalypse that was released after 2010 where the main character is a woman
Content Query (Semantic): apocalypse woman
Metadata Filter (Structured):
{'attribute': 'year', 'comparator': 'gt', 'value': 2010}
